In [1]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.ls_bip import laplace
from cardiac_electrophysiology.utils import analysis, visualization

In [ ]:
posterior_settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_laplace_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.05,
        tau=10,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-3,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

In [ ]:
posterior_builder = builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
map_estimate = np.load("../results/map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_estimate,
    posterior=posterior,
    additional_output=additional_output,
)
map_solution = posterior.parameter_to_solution_map.evaluate_forward(map_estimate)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=map_estimate,
    circular=False,
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=map_solution,
    circular=False,
)

In [ ]:
hessian_eigenvalues, hessian_eigenvectors = (
    laplace.compute_preconditioned_hessian_lowrank_approximation(
        posterior, map_estimate, map_solution, num_eigenvalues=200, oversampling_factor=50, seed=0
    )
)
prior_covariance_eigenvalues, prior_covariance_eigenvectors = (
    laplace.compute_prior_covariance_lowrank_approximation(
        posterior.prior, map_estimate, num_eigenvalues=1000, oversampling_factor=0, seed=0
    )
)
lowrank_data = {
    "hessian_eigenvalues": hessian_eigenvalues,
    "hessian_eigenvectors": hessian_eigenvectors,
    "prior_covariance_eigenvalues": prior_covariance_eigenvalues,
    "prior_covariance_eigenvectors": prior_covariance_eigenvectors,
}
np.savez("../results/lowrank_data.npz", **lowrank_data)

In [ ]:
lowrank_data = np.load("../results/lowrank_data.npz")
print(lowrank_data["hessian_eigenvectors"][:, 0] ** 2)
print(lowrank_data["prior_covariance_eigenvectors"][:, 0] ** 2)

[-0.48819398 -0.48799508 -0.48918702 ... -0.49700333 -0.49229445
 -0.47326916]
[-0.00787317 -0.00785823 -0.00787126 ... -0.00900381 -0.00819797
 -0.00743733]


In [ ]:
lowrank_data = np.load("../results/lowrank_data.npz")
laplace_approximation = laplace.LaplaceApproximation(
    hessian_eigenvalues=lowrank_data["hessian_eigenvalues"][:50],
    hessian_eigenvectors=lowrank_data["hessian_eigenvectors"][:, :50],
    prior_covariance_eigenvalues=lowrank_data["prior_covariance_eigenvalues"],
    prior_covariance_eigenvectors=lowrank_data["prior_covariance_eigenvectors"],
    map_estimate=map_estimate,
    prior=posterior.prior,
)
visualization.plot_eigenvalues(lowrank_data["hessian_eigenvalues"])
visualization.plot_eigenvalues(lowrank_data["prior_covariance_eigenvalues"])

In [ ]:
random_vector_size = posterior.prior.random_vector_size
rng = np.random.default_rng(seed=0)

samples = []
for _ in range(1000):
    random_vector = rng.standard_normal(random_vector_size)
    prior_sample = posterior.prior.apply_covariance_factorization(random_vector)
    samples.append(prior_sample)

variance = np.var(samples, axis=0)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=variance,
    circular=False,
)

In [ ]:
posterior_variance, prior_variance = laplace_approximation.compute_pointwise_variance(
    return_prior=True
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=prior_variance,
    circular=False,
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=posterior_variance,
    circular=False,
)